# Fetch missing abstracts (62 records)
Add `missing_abstracts.csv` as a dataset (or upload it), turn **Internet on**, then Run All. Download `abstracts_found.csv` from the Output tab and send it back.

In [ ]:

# Fetch missing abstracts for 62 records (Semantic Scholar, OpenAlex, Crossref). Runs in about 2 minutes.
import pandas as pd, requests, time, re, html, os
EMAIL = "your.email@example.org"
src = "missing_abstracts.csv"
for root, _, files in os.walk("/kaggle/input"):
    if "missing_abstracts.csv" in files: src = os.path.join(root, "missing_abstracts.csv")
df = pd.read_csv(src)
def clean(t): return re.sub(r"\s+", " ", html.unescape(re.sub(r"<[^>]+>", " ", t or ""))).strip()
def s2(doi):
    r = requests.get(f"https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}", params={"fields": "abstract,tldr"}, timeout=20)
    if r.ok:
        j = r.json(); a = j.get("abstract") or ((j.get("tldr") or {}).get("text"))
        return a, "s2" if j.get("abstract") else "s2-tldr"
    return None, None
def oa(doi):
    r = requests.get(f"https://api.openalex.org/works/doi:{doi}", params={"mailto": EMAIL}, timeout=20)
    if r.ok and r.json().get("abstract_inverted_index"):
        inv = r.json()["abstract_inverted_index"]; pos = {p: w for w, ps in inv.items() for p in ps}
        return " ".join(pos[k] for k in sorted(pos)), "openalex"
    return None, None
def cr(doi):
    r = requests.get(f"https://api.crossref.org/works/{doi}", params={"mailto": EMAIL}, timeout=20)
    if r.ok and r.json()["message"].get("abstract"): return clean(r.json()["message"]["abstract"]), "crossref"
    return None, None
out = []
for _, row in df.iterrows():
    doi = str(row.doi).replace("https://doi.org/", "")
    ab, srcname = None, None
    for f in (cr, s2, oa):
        try: ab, srcname = f(doi)
        except Exception as e: pass
        if ab and len(ab) > 150: break
        time.sleep(1.1)
    out.append({"rec_id": row.rec_id, "title": row.title, "abstract": clean(ab) if ab else "", "source": srcname or ""})
    print(row.rec_id, srcname, len(ab or ""))
res = pd.DataFrame(out); res.to_csv("/kaggle/working/abstracts_found.csv", index=False)
print("found:", (res.abstract.str.len() > 150).sum(), "of", len(res))
